[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/098.%20The%20Green%20Vehicle%20Routing%20Problem/P98-Tier-9_executed.ipynb)

---


# 98. The Green Vehicle Routing Problem

## Tier 9: QUBO Formulation and Quantum-Inspired Optimization

### Purpose and scope

This notebook formulates a small Green Vehicle Routing Problem (G-VRP) as a Quadratic Unconstrained Binary Optimization (QUBO) problem. It then applies simulated annealing and a classical state-vector simulation of the Quantum Approximate Optimization Algorithm (QAOA) to the same QUBO. The purpose is to show how routing decisions and selected constraints can be represented with binary variables and penalty terms.

The example has two customers, two vehicles, and two possible customer positions per vehicle. Node `0` is the depot, and nodes `1` and `2` are customers. The input distance and emissions matrices give the directed travel quantities between nodes; in this example, both matrices are symmetric. Customer demands and vehicle capacities are measured in the same load units. Route cost is reported at `$2` per distance unit, while emissions are reported in kilograms of CO₂. The coordinates used in the route plot are illustrative and do not determine the distance or emissions matrices.

### Model assumptions

- Every customer must be assigned to exactly one vehicle and one position.
- A vehicle can serve at most one customer at each position. A vehicle's occupied positions form a consecutive prefix: position `0` may be occupied while position `1` is empty, but position `1` cannot be occupied while position `0` is empty.
- Each route starts and ends at the depot. Empty vehicles are allowed.
- The capacity penalty is exact for capacity feasibility when the exactly-once assignment constraints hold in this two-customer instance. It penalizes a customer whose demand alone exceeds capacity and a pair assigned to the same vehicle whose combined demand exceeds capacity. The pairwise construction does not, in general, detect every overload involving three or more distinct customers; nor is it intended to account for repeated assignments in a vector that already violates the assignment constraints. The decoded solution checks capacity directly.
- The objective is a weighted sum of distance and emissions, using the specified weights of `0.5` for each. These weights define a scalarization convention for quantities with different units; QUBO energy is not a monetary cost or a physical measurement. Reported monetary cost and emissions are separate route metrics.
- This small demonstration does not model time windows, vehicle-dependent travel, charging infrastructure, traffic, or uncertainty. The state-vector QAOA simulation is classical and does not demonstrate quantum hardware performance or a quantum speedup.

### Method overview

The solution sequence is: define the instance and binary encoding; assemble the QUBO matrix from objective and penalty terms; inspect and test the matrix; search its binary state space with simulated annealing and QAOA; then interpret the decoded routes and constraint checks.

For customer `i`, position `p`, and vehicle `k`, the binary variable `x_{i,p,k}` is `1` exactly when that customer is assigned to that position on that vehicle. A binary vector `x` collects all these decisions. The objective is represented by a matrix `Q` such that the energy is `E(x) = x^T Q x`. Since `x` is binary, `x_i^2 = x_i`; this identity lets linear terms and quadratic interactions share one matrix representation. Penalties add an energy cost to assignments that violate modeled constraints.

For a symmetric `Q`, an off-diagonal coefficient `q` in the polynomial `E(x)` is stored as `q/2` in each of `Q[i, j]` and `Q[j, i]`. This convention ensures that `x^T Q x` contributes `q x_i x_j`, rather than twice that amount. Constants in penalty expansions do not affect which binary vector minimizes energy and are omitted from `Q`.

### Mathematical model represented by the QUBO

The binary encoding turns assignment and route structure into terms in one scalar energy. Define the assignment count for customer `i` and the occupancy of position `p` on vehicle `k` by

$$A_i = \sum_{p=0}^{P-1}\sum_{k=0}^{K-1} x_{i,p,k}, \qquad y_{p,k} = \sum_{i=1}^{C} x_{i,p,k}.$$

Here, `C`, `P`, and `K` are the numbers of customers, positions per vehicle, and vehicles. The assignment requirement is `A_i = 1`. Under the at-most-one-customer position constraint, `y_{p,k}` is binary. Define the weighted travel contribution for a directed leg from node `a` to node `b` as

$$c_{ab} = w_d d_{ab} + w_e e_{ab},$$

where `d_{ab}` and `e_{ab}` are the distance and emissions for that leg, and `w_d` and `w_e` are the distance and emissions weights. In the code, these quantities are stored in `distances`, `emissions`, `cost_weight`, and `emissions_weight`; `_weighted_travel` computes `c_{ab}`.

The travel objective accounts for travel from the depot to the first customer, travel between consecutive positions, and the return to the depot when a route ends. Let `y_{P,k}=0` represent the absence of a position after the final available position. The travel portion is

$$E_{\mathrm{travel}}(x) = \sum_{k=0}^{K-1}\sum_{i=1}^{C} c_{0i}x_{i,0,k} + \sum_{k=0}^{K-1}\sum_{p=0}^{P-2}\sum_{i=1}^{C}\sum_{j=1}^{C} c_{ij}x_{i,p,k}x_{j,p+1,k} + \sum_{k=0}^{K-1}\sum_{p=0}^{P-1}\sum_{i=1}^{C} c_{i0}x_{i,p,k}(1-y_{p+1,k}).$$

The last term charges a return leg when a customer is assigned to a position with no occupied successor. For the final position, the convention `y_{P,k}=0` makes that return unconditional whenever the final position is occupied. On feasible assignments, these terms match the complete depot-to-depot routes used to calculate route metrics.

The QUBO energy adds penalties for assignment, position occupancy, capacity, and route continuity:

$$\begin{aligned}
E(x) =\;& E_{\mathrm{travel}}(x)
+ \lambda_A\sum_{i=1}^{C}(A_i-1)^2
+ \lambda_P\sum_{k=0}^{K-1}\sum_{p=0}^{P-1}\sum_{1\leq i<j\leq C}x_{i,p,k}x_{j,p,k}\\
&+ \lambda_V\sum_{k=0}^{K-1}\left[\sum_{i:\,d_i>C_k}\sum_{p=0}^{P-1}x_{i,p,k} + \sum_{\substack{1\leq i<j\leq C\\d_i+d_j>C_k}}\sum_{p=0}^{P-1}\sum_{q=0}^{P-1}x_{i,p,k}x_{j,q,k}\right]
+ \lambda_C\sum_{k=0}^{K-1}\sum_{p=0}^{P-2}y_{p+1,k}(1-y_{p,k}).
\end{aligned}$$

Here, `d_i` is customer `i`'s demand, `C_k` is vehicle `k`'s capacity, and `\lambda_A`, `\lambda_P`, `\lambda_V`, and `\lambda_C` are the assignment, position, vehicle-capacity, and continuity penalty weights. The assignment penalty is zero exactly when each customer is selected once. The position penalty is positive when two customers occupy the same vehicle position. The capacity terms detect an individually over-capacity demand or, for two customers, an over-capacity pair on the same vehicle. The continuity term is positive when a later position is occupied while its immediately preceding position is empty; it therefore enforces the consecutive-prefix route structure. These terms are assembled by `_add_objective_terms`, `_add_assignment_constraints`, `_add_position_constraints`, `_add_capacity_constraints`, and `_add_continuity_constraints`.

The model uses penalty weights rather than hard constraints during the search. Consequently, a low-energy vector is not automatically feasible: the returned bitstring is decoded and its assignment, capacity, position, and continuity conditions are checked separately. The matrix construction uses the same binary decisions and penalty terms as the equations above, with constant terms omitted because they do not change the minimizing bitstring.

### Search methods and interpretation

Simulated annealing proposes single-bit flips. It accepts energy-decreasing moves and sometimes accepts energy-increasing moves with probability `exp(-Delta E / T)`, where `T` decreases through the specified temperature schedule. This allows exploration early in the search while favoring lower-energy states later. It is a heuristic search and does not certify a global optimum.

The QAOA routine uses the same QUBO energy as its diagonal cost Hamiltonian. For `n` binary variables, it simulates a state vector of length `2^n`, applies alternating cost-phase and mixer operations, and uses classical parameter optimization to minimize expected energy. It then samples bitstrings from the resulting state and reports the lowest-energy sample. This is a classical simulation of QAOA, not a run on a quantum processor. State-vector storage grows exponentially with the number of variables.

Feasibility is checked from the decoded binary assignment, independently of energy. A low-energy result is not automatically feasible: the penalty weights must sufficiently discourage violations, and heuristic searches may return an infeasible state. Compare routes, cost, emissions, energy, and the explicit feasibility report; do not interpret a comparison between these two runs as evidence of quantum advantage.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
from scipy.optimize import minimize

# A fixed seed makes the stochastic searches and sampled QAOA results reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class QUBOInstance:
    """Data and dimensions for a small QUBO Green VRP instance."""

    n_customers: int
    n_vehicles: int
    n_positions: int
    distances: np.ndarray
    emissions: np.ndarray
    vehicle_capacities: List[int]
    customer_demands: List[int]

    def __post_init__(self):
        if self.n_customers < 1 or self.n_vehicles < 1 or self.n_positions < 1:
            raise ValueError("The instance must have customers, vehicles, and positions.")
        if self.distances.shape != (self.n_customers + 1, self.n_customers + 1):
            raise ValueError("The distance matrix must include the depot and every customer.")
        if self.emissions.shape != self.distances.shape:
            raise ValueError("The emissions matrix must have the same shape as the distance matrix.")
        if len(self.vehicle_capacities) != self.n_vehicles:
            raise ValueError("Provide one capacity for each vehicle.")
        if len(self.customer_demands) != self.n_customers:
            raise ValueError("Provide one demand for each customer.")

        self.n_variables = self.n_customers * self.n_positions * self.n_vehicles

    def get_variable_index(self, customer: int, position: int, vehicle: int) -> int:
        """Map a customer-position-vehicle triple to a binary-vector index."""
        return (
            customer * self.n_positions * self.n_vehicles
            + position * self.n_vehicles
            + vehicle
        )


@dataclass
class QUBOSolution:
    """Decoded route metrics and feasibility information for a binary vector."""

    binary_vector: np.ndarray
    routes: List[List[int]]
    total_cost: float
    total_emissions: float
    total_distance: float
    energy: float
    is_feasible: bool
    constraint_violations: Dict[str, float]


class QUBO_GVRP:
    """Build and evaluate the QUBO model for the Green Vehicle Routing Problem."""

    def __init__(self, instance: QUBOInstance):
        self.instance = instance
        self.Q_matrix: Optional[np.ndarray] = None
        self.penalty_weights = {
            "assignment": 1000.0,
            "position": 1000.0,
            "vehicle": 1000.0,
            "continuity": 1000.0,
        }
        self.cost_weight = 0.5
        self.emissions_weight = 0.5

    @staticmethod
    def _add_quadratic_term(Q: np.ndarray, first: int, second: int, coefficient: float):
        """Store one polynomial term while preserving the symmetric-matrix convention."""
        if first == second:
            Q[first, first] += coefficient
        else:
            Q[first, second] += coefficient / 2
            Q[second, first] += coefficient / 2

    def _weighted_travel(self, from_node: int, to_node: int) -> float:
        """Return the weighted distance-emissions contribution of one travel leg."""
        distance = self.instance.distances[from_node, to_node]
        emissions = self.instance.emissions[from_node, to_node]
        return self.cost_weight * distance + self.emissions_weight * emissions

    def build_qubo_matrix(self) -> np.ndarray:
        """Assemble objective and penalty terms into a symmetric QUBO matrix."""
        if self.instance.n_customers != 2:
            raise ValueError(
                "The pairwise capacity penalty in this teaching model is exact for two customers."
            )

        n = self.instance.n_variables
        Q = np.zeros((n, n), dtype=float)

        print("Building QUBO matrix...")
        print(f"  Variables: {n}")
        print(f"  Customers: {self.instance.n_customers}")
        print(f"  Vehicles: {self.instance.n_vehicles}")
        print(f"  Positions: {self.instance.n_positions}")

        self._add_objective_terms(Q)
        self._add_assignment_constraints(Q)
        self._add_position_constraints(Q)
        self._add_capacity_constraints(Q)
        self._add_continuity_constraints(Q)

        self.Q_matrix = Q
        nonzero_count = np.count_nonzero(Q)
        sparsity = (1 - nonzero_count / (n * n)) * 100
        print(f"QUBO matrix built: {n} x {n}")
        print(f"  Non-zero elements: {nonzero_count}")
        print(f"  Sparsity: {sparsity:.1f}%")
        return Q

    def _add_objective_terms(self, Q: np.ndarray):
        """Add depot, consecutive-customer, and return-to-depot travel costs."""
        print("  Adding objective terms...")
        instance = self.instance

        for vehicle in range(instance.n_vehicles):
            for customer in range(instance.n_customers):
                first_idx = instance.get_variable_index(customer, 0, vehicle)
                depot_cost = self._weighted_travel(0, customer + 1)
                self._add_quadratic_term(Q, first_idx, first_idx, depot_cost)

                for position in range(instance.n_positions):
                    current_idx = instance.get_variable_index(customer, position, vehicle)
                    return_cost = self._weighted_travel(customer + 1, 0)

                    if position == instance.n_positions - 1:
                        self._add_quadratic_term(Q, current_idx, current_idx, return_cost)
                    else:
                        next_indices = [
                            instance.get_variable_index(next_customer, position + 1, vehicle)
                            for next_customer in range(instance.n_customers)
                        ]
                        self._add_quadratic_term(Q, current_idx, current_idx, return_cost)
                        for next_idx in next_indices:
                            self._add_quadratic_term(Q, current_idx, next_idx, -return_cost)

                    if position < instance.n_positions - 1:
                        for next_customer in range(instance.n_customers):
                            next_idx = instance.get_variable_index(
                                next_customer, position + 1, vehicle
                            )
                            travel_cost = self._weighted_travel(
                                customer + 1, next_customer + 1
                            )
                            self._add_quadratic_term(Q, current_idx, next_idx, travel_cost)

        print("    Objective terms added")

    def _add_assignment_constraints(self, Q: np.ndarray):
        """Penalize assignments in which a customer is selected other than once."""
        print("  Adding assignment constraints...")
        penalty = self.penalty_weights["assignment"]

        for customer in range(self.instance.n_customers):
            indices = [
                self.instance.get_variable_index(customer, position, vehicle)
                for position in range(self.instance.n_positions)
                for vehicle in range(self.instance.n_vehicles)
            ]

            # Expanding (sum(x) - 1)^2 gives linear coefficient -1 and pair coefficient 2.
            for index in indices:
                self._add_quadratic_term(Q, index, index, -penalty)
            for first in range(len(indices)):
                for second in range(first + 1, len(indices)):
                    self._add_quadratic_term(
                        Q, indices[first], indices[second], 2 * penalty
                    )

        print("    Assignment constraints added")

    def _add_position_constraints(self, Q: np.ndarray):
        """Penalize two customers occupying the same position on one vehicle."""
        print("  Adding position constraints...")
        penalty = self.penalty_weights["position"]

        for vehicle in range(self.instance.n_vehicles):
            for position in range(self.instance.n_positions):
                indices = [
                    self.instance.get_variable_index(customer, position, vehicle)
                    for customer in range(self.instance.n_customers)
                ]
                for first in range(len(indices)):
                    for second in range(first + 1, len(indices)):
                        self._add_quadratic_term(
                            Q, indices[first], indices[second], penalty
                        )

        print("    Position constraints added")

    def _add_capacity_constraints(self, Q: np.ndarray):
        """Penalize individual or paired assignments that exceed vehicle capacity."""
        print("  Adding capacity constraints...")
        penalty = self.penalty_weights["vehicle"]
        instance = self.instance

        for vehicle in range(instance.n_vehicles):
            capacity = instance.vehicle_capacities[vehicle]
            for customer in range(instance.n_customers):
                demand = instance.customer_demands[customer]
                if demand > capacity:
                    for position in range(instance.n_positions):
                        index = instance.get_variable_index(customer, position, vehicle)
                        self._add_quadratic_term(Q, index, index, penalty)

            for first_customer in range(instance.n_customers):
                for second_customer in range(first_customer + 1, instance.n_customers):
                    combined_demand = (
                        instance.customer_demands[first_customer]
                        + instance.customer_demands[second_customer]
                    )
                    if combined_demand > capacity:
                        for first_position in range(instance.n_positions):
                            first_idx = instance.get_variable_index(
                                first_customer, first_position, vehicle
                            )
                            for second_position in range(instance.n_positions):
                                second_idx = instance.get_variable_index(
                                    second_customer, second_position, vehicle
                                )
                                self._add_quadratic_term(
                                    Q, first_idx, second_idx, penalty
                                )

        print("    Capacity constraints added")

    def _add_continuity_constraints(self, Q: np.ndarray):
        """Penalize an occupied position whose immediately preceding position is empty."""
        print("  Adding continuity constraints...")
        penalty = self.penalty_weights["continuity"]
        instance = self.instance

        for vehicle in range(instance.n_vehicles):
            for position in range(instance.n_positions - 1):
                current_indices = [
                    instance.get_variable_index(customer, position, vehicle)
                    for customer in range(instance.n_customers)
                ]
                next_indices = [
                    instance.get_variable_index(customer, position + 1, vehicle)
                    for customer in range(instance.n_customers)
                ]

                # y_(p+1)(1-y_p) penalizes a later occupied position with no predecessor.
                for next_idx in next_indices:
                    self._add_quadratic_term(Q, next_idx, next_idx, penalty)
                    for current_idx in current_indices:
                        self._add_quadratic_term(Q, current_idx, next_idx, -penalty)

        print("    Continuity constraints added")

    def calculate_energy(self, binary_vector: np.ndarray) -> float:
        """Evaluate x-transpose-Q-x for a binary decision vector."""
        if self.Q_matrix is None:
            raise ValueError("Build the QUBO matrix before calculating energy.")
        vector = np.asarray(binary_vector, dtype=float)
        if vector.shape != (self.instance.n_variables,):
            raise ValueError("The binary vector has the wrong number of variables.")
        return float(vector @ self.Q_matrix @ vector)

    def decode_solution(self, binary_vector: np.ndarray) -> QUBOSolution:
        """Decode selected variables into routes and calculate complete route metrics."""
        vector = np.asarray(binary_vector, dtype=int)
        if vector.shape != (self.instance.n_variables,):
            raise ValueError("The binary vector has the wrong number of variables.")

        routes = []
        total_cost = 0.0
        total_emissions = 0.0
        total_distance = 0.0

        for vehicle in range(self.instance.n_vehicles):
            route = []
            for position in range(self.instance.n_positions):
                for customer in range(self.instance.n_customers):
                    index = self.instance.get_variable_index(customer, position, vehicle)
                    if vector[index] == 1:
                        route.append(customer + 1)

            if route:
                routes.append(route)
                full_route = [0] + route + [0]
                for from_node, to_node in zip(full_route[:-1], full_route[1:]):
                    distance = self.instance.distances[from_node, to_node]
                    emissions = self.instance.emissions[from_node, to_node]
                    total_distance += distance
                    total_emissions += emissions
                    total_cost += 2.0 * distance

        violations = self._check_constraints(vector)
        return QUBOSolution(
            binary_vector=vector.copy(),
            routes=routes,
            total_cost=total_cost,
            total_emissions=total_emissions,
            total_distance=total_distance,
            energy=self.calculate_energy(vector),
            is_feasible=not violations,
            constraint_violations=violations,
        )

    def _check_constraints(self, binary_vector: np.ndarray) -> Dict[str, float]:
        """Report assignment, position, capacity, and route-continuity violations."""
        vector = np.asarray(binary_vector)
        if vector.shape != (self.instance.n_variables,):
            raise ValueError("The binary vector has the wrong number of variables.")
        if not np.all(np.isin(vector, [0, 1])):
            raise ValueError("Constraint checks require a binary vector.")

        instance = self.instance
        violations: Dict[str, float] = {}

        for customer in range(instance.n_customers):
            assignments = sum(
                vector[instance.get_variable_index(customer, position, vehicle)]
                for position in range(instance.n_positions)
                for vehicle in range(instance.n_vehicles)
            )
            if assignments != 1:
                violations[f"customer_{customer}_assignment"] = float(abs(assignments - 1))

        for vehicle in range(instance.n_vehicles):
            total_demand = sum(
                vector[instance.get_variable_index(customer, position, vehicle)]
                * instance.customer_demands[customer]
                for customer in range(instance.n_customers)
                for position in range(instance.n_positions)
            )
            if total_demand > instance.vehicle_capacities[vehicle]:
                violations[f"vehicle_{vehicle}_capacity"] = float(
                    total_demand - instance.vehicle_capacities[vehicle]
                )

            for position in range(instance.n_positions):
                occupancy = sum(
                    vector[instance.get_variable_index(customer, position, vehicle)]
                    for customer in range(instance.n_customers)
                )
                if occupancy > 1:
                    violations[f"vehicle_{vehicle}_position_{position}"] = float(
                        occupancy - 1
                    )

            for position in range(instance.n_positions - 1):
                current_occupancy = sum(
                    vector[instance.get_variable_index(customer, position, vehicle)]
                    for customer in range(instance.n_customers)
                )
                next_occupancy = sum(
                    vector[instance.get_variable_index(customer, position + 1, vehicle)]
                    for customer in range(instance.n_customers)
                )
                if current_occupancy == 0 and next_occupancy > 0:
                    violations[f"vehicle_{vehicle}_continuity_{position}"] = 1.0

        return violations

In [ ]:
# Define the small two-customer instance used throughout the notebook.
print("=" * 60)
print("GREEN VRP - QUBO FORMULATION SETUP")
print("=" * 60)

# Node 0 is the depot; nodes 1 and 2 are customers. Distances are in distance units.
distances = np.array(
    [
        [0, 10, 12],
        [10, 0, 8],
        [12, 8, 0],
    ],
    dtype=float,
)

# Emissions are in kilograms of CO2 for the corresponding travel leg.
emissions = np.array(
    [
        [0, 6, 10],
        [6, 0, 5],
        [10, 5, 0],
    ],
    dtype=float,
)

instance = QUBOInstance(
    n_customers=2,
    n_vehicles=2,
    n_positions=2,
    distances=distances,
    emissions=emissions,
    vehicle_capacities=[50, 50],
    customer_demands=[20, 30],
)

print("QUBO instance created:")
print(f"  Customers: {instance.n_customers}")
print(f"  Vehicles: {instance.n_vehicles}")
print(f"  Positions per vehicle: {instance.n_positions}")
print(f"  Binary variables: {instance.n_variables}")
print(f"  Vehicle capacities: {instance.vehicle_capacities}")
print(f"  Customer demands: {instance.customer_demands}")
print("\nDistance matrix:")
print(distances)
print("\nEmissions matrix (kg CO2 per leg):")
print(emissions)

qubo_solver = QUBO_GVRP(instance)
print("\nQUBO solver created:")
print(f"  Penalty weights: {qubo_solver.penalty_weights}")
print(f"  Distance weight: {qubo_solver.cost_weight}")
print(f"  Emissions weight: {qubo_solver.emissions_weight}")
print("\nReady to construct the QUBO matrix.")

In [ ]:
# Construct the matrix and inspect its basic structure before searching it.
Q_matrix = qubo_solver.build_qubo_matrix()

print("\n" + "=" * 60)
print("QUBO MATRIX ANALYSIS")
print("=" * 60)

nonzero_count = np.count_nonzero(Q_matrix)
sparsity = (1 - nonzero_count / Q_matrix.size) * 100
print(f"Matrix shape: {Q_matrix.shape}")
print(f"Non-zero elements: {nonzero_count}")
print(f"Sparsity: {sparsity:.1f}%")
print(f"Symmetric: {np.allclose(Q_matrix, Q_matrix.T)}")
print(f"Minimum entry: {Q_matrix.min():.2f}")
print(f"Maximum entry: {Q_matrix.max():.2f}")
print(f"Mean entry: {Q_matrix.mean():.2f}")
print("\nQUBO matrix:")
print(Q_matrix)

# The all-zero vector is not a valid routing assignment, but its energy is a useful reference.
zero_vector = np.zeros(instance.n_variables, dtype=int)
print(f"\nAll-zero vector energy: {qubo_solver.calculate_energy(zero_vector):.2f}")

# Each customer uses a different vehicle at its first position, forming two valid one-customer routes.
test_vector = np.zeros(instance.n_variables, dtype=int)
test_vector[instance.get_variable_index(0, 0, 0)] = 1
test_vector[instance.get_variable_index(1, 0, 1)] = 1
test_solution = qubo_solver.decode_solution(test_vector)

assert test_solution.is_feasible
assert np.isclose(test_solution.energy, qubo_solver.calculate_energy(test_vector))
print("\nTest assignment:")
print(f"  Energy: {test_solution.energy:.2f}")
print(f"  Routes: {test_solution.routes}")
print(f"  Cost: ${test_solution.total_cost:.2f}")
print(f"  Distance: {test_solution.total_distance:.2f} distance units")
print(f"  Emissions: {test_solution.total_emissions:.2f} kg CO2")
print(f"  Feasible: {test_solution.is_feasible}")
print(f"  Constraint violations: {test_solution.constraint_violations}")

In [ ]:
class QuantumInspiredSolver:
    """Classical simulated annealing and state-vector QAOA for a small QUBO."""

    def __init__(self, qubo_solver: QUBO_GVRP):
        if qubo_solver.Q_matrix is None:
            raise ValueError("Build the QUBO matrix before creating the search solver.")
        self.qubo_solver = qubo_solver

    def simulated_annealing(
        self,
        max_iterations: int = 1000,
        initial_temp: float = 10.0,
        final_temp: float = 0.01,
    ) -> Tuple[QUBOSolution, List[float]]:
        """Search the binary state space using a decreasing-temperature schedule."""
        if max_iterations < 1 or initial_temp <= 0 or final_temp <= 0:
            raise ValueError("Iterations and both temperatures must be positive.")

        print("\n=== Simulated Annealing ===")
        print(f"Maximum iterations: {max_iterations}")
        print(f"Temperature range: {initial_temp} to {final_temp}")
        n = self.qubo_solver.instance.n_variables
        current_vector = np.random.randint(0, 2, n)
        current_energy = self.qubo_solver.calculate_energy(current_vector)
        best_vector = current_vector.copy()
        best_energy = current_energy
        temperature_schedule = np.linspace(initial_temp, final_temp, max_iterations)
        energy_history = [current_energy]

        for iteration, temperature in enumerate(temperature_schedule, start=1):
            candidate = current_vector.copy()
            flip_index = np.random.randint(0, n)
            candidate[flip_index] = 1 - candidate[flip_index]
            candidate_energy = self.qubo_solver.calculate_energy(candidate)
            energy_change = candidate_energy - current_energy

            # Uphill moves are probabilistic so the search can escape some local minima.
            if energy_change < 0 or np.random.random() < np.exp(-energy_change / temperature):
                current_vector = candidate
                current_energy = candidate_energy
                if current_energy < best_energy:
                    best_vector = current_vector.copy()
                    best_energy = current_energy

            energy_history.append(current_energy)
            if iteration % 100 == 0:
                print(
                    f"Iteration {iteration}/{max_iterations}: "
                    f"current energy = {current_energy:.2f}, "
                    f"best energy = {best_energy:.2f}, temperature = {temperature:.3f}"
                )

        best_solution = self.qubo_solver.decode_solution(best_vector)
        print("Simulated annealing complete:")
        print(f"  Best energy: {best_solution.energy:.2f}")
        print(f"  Route cost: ${best_solution.total_cost:.2f}")
        print(f"  Emissions: {best_solution.total_emissions:.2f} kg CO2")
        print(f"  Feasible: {best_solution.is_feasible}")
        return best_solution, energy_history

    @staticmethod
    def _apply_mixer(state: np.ndarray, beta: float, n_qubits: int) -> np.ndarray:
        """Apply exp(-i beta X) to each qubit in a state vector."""
        mixed_state = state.copy()
        cosine = np.cos(beta)
        sine = -1j * np.sin(beta)

        for qubit in range(n_qubits):
            bit_mask = 1 << qubit
            for basis_index in range(len(mixed_state)):
                if basis_index & bit_mask == 0:
                    paired_index = basis_index | bit_mask
                    amplitude_zero = mixed_state[basis_index]
                    amplitude_one = mixed_state[paired_index]
                    mixed_state[basis_index] = (
                        cosine * amplitude_zero + sine * amplitude_one
                    )
                    mixed_state[paired_index] = (
                        sine * amplitude_zero + cosine * amplitude_one
                    )

        return mixed_state

    def _qaoa_state(self, parameters: np.ndarray, layers: int) -> np.ndarray:
        """Apply alternating QUBO cost and X-mixer unitaries to a uniform state."""
        n = self.qubo_solver.instance.n_variables
        state_count = 1 << n
        basis_indices = np.arange(state_count, dtype=np.uint64)
        energies = np.empty(state_count, dtype=float)

        # Each basis index encodes one binary vector, whose diagonal energy is its QUBO value.
        for basis_index in range(state_count):
            vector = np.fromiter(
                ((basis_index >> bit) & 1 for bit in range(n)),
                dtype=int,
                count=n,
            )
            energies[basis_index] = self.qubo_solver.calculate_energy(vector)

        state = np.full(state_count, 1 / np.sqrt(state_count), dtype=complex)
        for layer in range(layers):
            gamma = parameters[layer]
            beta = parameters[layers + layer]
            state *= np.exp(-1j * gamma * energies)
            state = self._apply_mixer(state, beta, n)

        return state

    def quantum_approximate_optimization(
        self,
        layers: int = 3,
        max_iterations: int = 100,
        n_samples: int = 1000,
    ) -> QUBOSolution:
        """Optimize QAOA angles classically and select the best sampled bitstring."""
        if layers < 1 or max_iterations < 1 or n_samples < 1:
            raise ValueError("Layers, optimization iterations, and samples must be positive.")

        print("\n=== Classical State-Vector QAOA Simulation ===")
        print(f"Layers: {layers}")
        print(f"Classical optimizer iterations: {max_iterations}")
        n = self.qubo_solver.instance.n_variables
        initial_parameters = np.concatenate(
            [
                np.random.uniform(0, 2 * np.pi, layers),
                np.random.uniform(0, np.pi, layers),
            ]
        )

        def expected_energy(parameters: np.ndarray) -> float:
            state = self._qaoa_state(parameters, layers)
            probabilities = np.abs(state) ** 2
            energies = np.empty(len(probabilities), dtype=float)
            for basis_index in range(len(probabilities)):
                vector = np.fromiter(
                    ((basis_index >> bit) & 1 for bit in range(n)),
                    dtype=int,
                    count=n,
                )
                energies[basis_index] = self.qubo_solver.calculate_energy(vector)
            return float(probabilities @ energies)

        result = minimize(
            expected_energy,
            initial_parameters,
            method="COBYLA",
            options={"maxiter": max_iterations},
        )
        final_state = self._qaoa_state(result.x, layers)
        probabilities = np.abs(final_state) ** 2
        probabilities /= probabilities.sum()
        sampled_indices = np.random.choice(
            len(probabilities), size=n_samples, p=probabilities
        )

        best_vector = None
        best_energy = float("inf")
        for basis_index in sampled_indices:
            vector = np.fromiter(
                ((basis_index >> bit) & 1 for bit in range(n)),
                dtype=int,
                count=n,
            )
            energy = self.qubo_solver.calculate_energy(vector)
            if energy < best_energy:
                best_vector = vector
                best_energy = energy

        best_solution = self.qubo_solver.decode_solution(best_vector)
        print(f"Classical angle optimization complete: {result.message}")
        print(f"  Best sampled energy: {best_solution.energy:.2f}")
        print(f"  Route cost: ${best_solution.total_cost:.2f}")
        print(f"  Emissions: {best_solution.total_emissions:.2f} kg CO2")
        print(f"  Feasible: {best_solution.is_feasible}")
        return best_solution


# Both searches operate on the same QUBO matrix and use the established random seed.
solver = QuantumInspiredSolver(qubo_solver)
sa_solution, sa_energy_history = solver.simulated_annealing(
    max_iterations=1000,
    initial_temp=10.0,
    final_temp=0.01,
)
qaoa_solution = solver.quantum_approximate_optimization(
    layers=3,
    max_iterations=100,
    n_samples=1000,
)

In [ ]:
# Visualize the matrix, annealing trace, decoded routes, and feasibility outcomes.
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("QUBO Green VRP: Classical Search Results", fontsize=16, fontweight="bold")

ax1 = axes[0, 0]
image = ax1.imshow(Q_matrix, cmap="viridis", aspect="auto")
ax1.set_title("QUBO Matrix")
ax1.set_xlabel("Variable index")
ax1.set_ylabel("Variable index")
fig.colorbar(image, ax=ax1, label="Matrix entry")

ax2 = axes[0, 1]
ax2.plot(range(len(sa_energy_history)), sa_energy_history, color="tab:blue", linewidth=1.5)
ax2.set_title("Simulated Annealing Energy Trace")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("QUBO energy")
ax2.grid(True, alpha=0.3)
minimum_index = int(np.argmin(sa_energy_history))
ax2.annotate(
    f"Best: {sa_energy_history[minimum_index]:.2f}",
    xy=(minimum_index, sa_energy_history[minimum_index]),
    xytext=(5, 8),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "black"},
)

methods = ["Simulated annealing", "State-vector QAOA"]
solutions = [sa_solution, qaoa_solution]
ax3 = axes[0, 2]
x_positions = np.arange(len(methods))
bar_width = 0.35
ax3.bar(
    x_positions - bar_width / 2,
    [solution.total_cost for solution in solutions],
    bar_width,
    label="Route cost ($)",
)
ax3.bar(
    x_positions + bar_width / 2,
    [solution.total_emissions for solution in solutions],
    bar_width,
    label="Emissions (kg CO2)",
)
ax3.set_title("Decoded Route Metrics")
ax3.set_xlabel("Search method")
ax3.set_ylabel("Metric value (different units)")
ax3.set_xticks(x_positions)
ax3.set_xticklabels(methods, rotation=15, ha="right")
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

ax4 = axes[1, 0]
coordinates = {0: (0, 0), 1: (10, 0), 2: (12, 8)}
ax4.scatter(*coordinates[0], s=180, marker="s", color="black", label="Depot", zorder=4)
ax4.scatter(
    [coordinates[1][0], coordinates[2][0]],
    [coordinates[1][1], coordinates[2][1]],
    s=110,
    marker="o",
    color="tab:blue",
    label="Customer",
    zorder=4,
)
for customer, (x_coord, y_coord) in coordinates.items():
    ax4.annotate(str(customer), (x_coord, y_coord), xytext=(5, 5), textcoords="offset points")

route_colors = ["tab:orange", "tab:green", "tab:red", "tab:purple"]
for route_index, route in enumerate(sa_solution.routes):
    full_route = [0] + route + [0]
    route_color = route_colors[route_index % len(route_colors)]
    route_label = f"Annealing route {route_index + 1}"
    for leg_index, (from_node, to_node) in enumerate(zip(full_route[:-1], full_route[1:])):
        start = coordinates[from_node]
        end = coordinates[to_node]
        ax4.plot(
            [start[0], end[0]],
            [start[1], end[1]],
            color=route_color,
            linewidth=2,
            alpha=0.8,
            label=route_label if leg_index == 0 else None,
        )
ax4.set_title("Simulated Annealing Routes (Illustrative Coordinates)")
ax4.set_xlabel("X coordinate (illustrative units)")
ax4.set_ylabel("Y coordinate (illustrative units)")
ax4.grid(True, alpha=0.3)
ax4.legend()

ax5 = axes[1, 1]
violation_counts = [len(solution.constraint_violations) for solution in solutions]
bar_colors = ["tab:green" if count == 0 else "tab:red" for count in violation_counts]
bars = ax5.bar(methods, violation_counts, color=bar_colors, alpha=0.8)
ax5.set_title("Constraint Violations")
ax5.set_ylabel("Number of violated constraint groups")
ax5.tick_params(axis="x", rotation=15)
ax5.grid(True, axis="y", alpha=0.3)
for bar, solution in zip(bars, solutions):
    label = "Feasible" if solution.is_feasible else "Infeasible"
    ax5.annotate(
        label,
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
    )

ax6 = axes[1, 2]
metric_names = ["Cost", "Emissions", "Distance", "Feasibility"]
metric_values = []
for solution in solutions:
    metric_values.append(
        [
            solution.total_cost,
            solution.total_emissions,
            solution.total_distance,
            float(solution.is_feasible),
        ]
    )
normalized_values = np.zeros((len(solutions), len(metric_names)))
for metric_index in range(3):
    scale = max(values[metric_index] for values in metric_values)
    if scale > 0:
        normalized_values[:, metric_index] = [
            values[metric_index] / scale * 100 for values in metric_values
        ]
normalized_values[:, 3] = [values[3] * 100 for values in metric_values]
metric_positions = np.arange(len(metric_names))
ax6.bar(
    metric_positions - bar_width / 2,
    normalized_values[0],
    bar_width,
    label="Simulated annealing",
)
ax6.bar(
    metric_positions + bar_width / 2,
    normalized_values[1],
    bar_width,
    label="State-vector QAOA",
)
ax6.set_title("Scaled Metrics for Visual Comparison")
ax6.set_xlabel("Metric")
ax6.set_ylabel("Relative score (0-100; lower is not always better)")
ax6.set_xticks(metric_positions)
ax6.set_xticklabels(metric_names)
ax6.legend()
ax6.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summarize the model and the two heuristic search results without assuming either is optimal.
print("\n" + "=" * 60)
print("QUBO FORMULATION ANALYSIS")
print("=" * 60)

print("\n1. QUBO representation:")
print(f"  Binary variables: {instance.n_variables}")
print(f"  Matrix dimensions: {Q_matrix.shape}")
print(f"  Matrix sparsity: {(1 - np.count_nonzero(Q_matrix) / Q_matrix.size) * 100:.1f}%")
print(f"  Symmetric matrix: {np.allclose(Q_matrix, Q_matrix.T)}")

print("\n2. Search results:")
for method, solution in zip(methods, solutions):
    print(f"  {method}:")
    print(f"    QUBO energy: {solution.energy:.2f}")
    print(f"    Route cost: ${solution.total_cost:.2f}")
    print(f"    Distance: {solution.total_distance:.2f} distance units")
    print(f"    Emissions: {solution.total_emissions:.2f} kg CO2")
    print(f"    Routes: {solution.routes}")
    print(f"    Feasible: {solution.is_feasible}")
    print(f"    Constraint violations: {solution.constraint_violations}")

print("\n3. Search-space size and computational representation:")
print(f"  Binary variables: {instance.n_variables}")
print(f"  Possible bitstrings: 2^{instance.n_variables} = {2 ** instance.n_variables:,}")
print("  Exhaustive evaluation grows exponentially in the number of binary variables.")
print("  Evaluating x-transpose-Q-x directly requires quadratic work in the variable count.")
print(
    "  A state-vector simulation stores 2^n amplitudes, so it is intended here only "
    "for a small demonstration."
)

print("\n4. Model interpretation:")
print("  The weighted travel objective includes distance and emissions on complete depot-to-depot routes.")
print("  Penalty terms encode customer assignment, position, two-customer capacity, and continuity rules.")
print("  Feasibility is checked from the decoded assignment, separately from the QUBO energy.")
print("  A heuristic result is a candidate solution, not a proof of global optimality.")

print("\n5. Method limitations:")
print("  Simulated annealing is a classical stochastic heuristic whose result depends on its schedule and run.")
print("  QAOA here is simulated classically with a finite state vector and classical angle optimization.")
print("  This notebook does not run on quantum hardware or establish a quantum advantage.")
print("  The pairwise capacity QUBO is scoped to the two-customer instance used in this example.")

print("\n6. How to read the comparison:")
print("  Compare energy only for the shared QUBO objective and penalty model.")
print("  Compare route cost, distance, and emissions using their stated units.")
print("  Treat feasibility as a separate requirement when interpreting either search result.")

## Key insights

1. **Binary encoding:** The variable `x_{i,p,k}` records whether customer `i` is assigned to position `p` on vehicle `k`. The encoding gives the QUBO a direct representation of the routing decisions in this small instance.

2. **Penalty modeling:** Assignment, position, capacity, and continuity requirements contribute penalty terms to the same quadratic energy as travel. The continuity term penalizes an occupied position whose immediately preceding position is empty, so occupied positions form a consecutive prefix. Penalty weights must be chosen with care: they should discourage violations without making numerical optimization unnecessarily poorly scaled. Feasibility is still checked explicitly after decoding.

3. **Objective and route metrics:** QUBO energy includes weighted distance and emissions plus penalties. Monetary route cost and emissions are reported separately from energy, and a low energy does not by itself imply a feasible or globally optimal route.

4. **Classical search and QAOA simulation:** Simulated annealing searches by probabilistic bit flips. The QAOA routine applies alternating cost and mixer operations to a classical state vector, optimizes circuit angles classically, and samples candidate bitstrings. Neither search is guaranteed to find a global optimum in this notebook.

5. **Scaling and evidence:** A state vector for `n` binary variables has `2^n` amplitudes. This exponential storage requirement limits classical simulation to small instances. The notebook demonstrates a formulation and two search procedures; it does not show quantum hardware performance or establish a speedup.

## Appropriate use and limitations

This formulation is useful for learning how a small routing model can be expressed as QUBO and how penalty-based feasibility can be assessed. It is also useful for controlled experiments on tiny instances where all variables and matrix terms can be inspected.

The capacity penalty in this implementation represents capacity feasibility exactly for valid exactly-once assignments in the two-customer instance. More customers can create overloads that depend on three or more assignments, which this pairwise construction does not represent in general. Real routing applications also need additional operational details, such as time windows, fleet-specific constraints, and charging decisions, when those features matter.

## Summary

The notebook maps a small Green Vehicle Routing Problem to a symmetric QUBO matrix, adds travel and constraint terms, and applies simulated annealing and a classical state-vector QAOA simulation. The decoded routes are evaluated using complete depot-to-depot distance, cost, and emissions, while constraint feasibility is reported separately. The resulting numerical candidates should be interpreted within the stated small-instance scope and should not be treated as evidence of quantum advantage.